# Homework: AI Orchestration with Kestra

[Homework instructions](homework.md)

## Question 1: Context Engineering

### ChatGPT

```yaml
id: nyc_taxi_csv_to_bigquery
namespace: zoomcamp

variables:
  project_id: "{{ envs.GCP_PROJECT_ID }}"
  dataset: ny_taxi
  table: yellow_tripdata_2024_01
  bucket: "{{ envs.GCP_BUCKET }}"
  file: yellow_tripdata_2024-01.csv.gz

tasks:
  - id: download_csv
    type: io.kestra.plugin.core.http.Download
    uri: "https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/{{ vars.file }}"

  - id: create_dataset
    type: io.kestra.plugin.gcp.bigquery.Query
    projectId: "{{ vars.project_id }}"
    serviceAccount: "{{ secret('GCP_CREDS') }}"
    sql: |
      CREATE SCHEMA IF NOT EXISTS `{{ vars.project_id }}.{{ vars.dataset }}`;

  - id: load_to_bigquery
    type: io.kestra.plugin.gcp.bigquery.Load
    from: "{{ outputs.download_csv.uri }}"
    destinationTable: "{{ vars.project_id }}.{{ vars.dataset }}.{{ vars.table }}"
    format: CSV
    csvOptions:
      fieldDelimiter: ","
      skipLeadingRows: 1
      allowQuotedNewLines: true
    autodetect: true
    writeDisposition: WRITE_TRUNCATE
```


### Kestra's AI Copilot

```yaml
id: nyc_taxi_data_load
namespace: dev.load

tasks:
  - id: download_taxi_data
    type: io.kestra.plugin.core.http.Download
    uri: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.csv

  - id: load_to_bigquery
    type: io.kestra.plugin.gcp.bigquery.Load
    from: "{{ outputs.download_taxi_data.uri }}"
    destinationTable: "your_project_id.your_dataset_id.your_table_id" # Replace with your actual project, dataset, and table
    format: CSV
    autodetect: true
    csvOptions:
      skipLeadingRows: 1 # Assuming the CSV has a header row
```

### Question: After trying the same prompt in ChatGPT vs Kestra's AI Copilot, what is the primary reason AI Copilot generates better Kestra flows?

- AI Copilot uses a more powerful model
- -> **AI Copilot has access to current Kestra plugin documentation** <-
- AI Copilot uses more tokens
- AI Copilot has internet access



Honestly, both seem reasonable. The one from ChatGPT is more explicit by defining variables, schema definition, and additional properties.

Note: At the time of writing (2026-07-02), ChatGPT can't be used in a private browser window anymore, which the instructions ask for.

## Question 2: RAG vs No RAG

### RAG

INFO 2026-06-30T23:53:06.439537Z ✅ RAG Response (with retrieved context):
Kestra 1.1 introduced several major features, including:

1.  **New Filters**: Redesigned UI filters with improved usability for tracking executions, monitoring workflow states, and analyzing historical runs.
2.  **No-Code Dashboard Editor**: Users can now create and edit custom dashboards using a no-code, multi-panel editor, building visualizations and charts without writing YAML.
3.  **Human Task**: This new feature allows Enterprise Edition users to add manual approval steps to workflows, pausing execution until designated users or groups approve it.
4.  **Multi-Agent AI Systems**: Kestra's AI agents can now use other AI agents as tools, enabling sophisticated multi-agent orchestration where a primary agent can delegate subtasks to specialized expert agents.
5.  **Fix with AI**: When task runs fail, Kestra 1.1 provides AI-powered suggestions to help diagnose and resolve issues, offering intelligent recommendations for fixes.

🎉 Note that this response is detailed, accurate, and grounded in the actual release documentation. Compare this with the output from 1_chat_without_rag.yaml!

### No RAG

INFO 2026-06-30T23:49:45.565689Z ❌ Response WITHOUT RAG (no retrieved context):
Kestra 1.1 introduced several exciting features that significantly enhanced its capabilities. Here are 5 major features with brief descriptions:

1.  **Conditions on Triggers:** This was a highly anticipated feature that allows users to define specific conditions that must be met *before* a trigger actually fires a flow run.
    *   **Description:** Instead of a trigger always running a flow when its condition (e.g., file arrival, schedule time) is met, you can now add a secondary set of conditions (e.g., a specific value in the file content, an external system's status) that must also be true. If the conditions are not met, the trigger is skipped, preventing unnecessary flow runs and providing more precise control over automation.

2.  **Plugin Configuration on the UI:** This feature greatly simplified the management and configuration of Kestra plugins, making it accessible to a wider audience.
    *   **Description:** Previously, configuring plugins often involved editing YAML files directly or dealing with complex setup procedures. Kestra 1.1 introduced the ability to configure installed plugins directly within the K Kestra UI. This includes setting up connections, credentials, and other parameters through intuitive forms, significantly improving usability and reducing the learning curve for integrating external systems.

3.  **Namespace-level Variables:** This feature brought a new level of organization and reusability to Kestra flows.
    *   **Description:** Before 1.1, variables were typically defined at the flow level or globally. Namespace-level variables allow users to define variables that are accessible to all flows within a specific namespace. This is incredibly useful for managing environment-specific configurations (e.g., API keys, database connection strings) or common parameters that many flows in a logical grouping might share, promoting consistency and reducing redundancy.

4.  **Flow-level Input Default Values:** This enhancement provided more flexibility and robustness to flow definitions, especially when dealing with optional inputs.
    *   **Description:** Users can now specify default values for input parameters at the flow level. If an input is not provided when a flow is triggered (either manually or via a trigger), Kestra will automatically use the defined default value instead of failing or requiring the input to be explicitly passed. This makes flows more resilient to missing inputs and simplifies their invocation in many scenarios.

5.  **New Plugin: Microsoft Teams:** This expanded Kestra's integration capabilities, allowing for better notification and collaboration workflows.
    *   **Description:** Kestra 1.1 introduced a dedicated plugin for Microsoft Teams. This plugin enables Kestra flows to send messages, notifications, and adaptive cards directly to Microsoft Teams channels. This is invaluable for communicating flow status, alerts, or results to relevant teams, integrating Kestra more seamlessly into enterprise communication platforms.

🤔 Did you notice that this response seems to be:
- Incorrect?
- Vague/generic?
- Listing features that haven't been added in exactly this version but rather a long time ago?

👉 This is why context matters! Run `2_chat_with_rag.yaml` to see the accurate, context-grounded response.

### Question: The non-RAG response about Kestra 1.1 features is best described as:

- Accurate and specific, matching the actual release notes
- -> **Vague, generic, or fabricated — the model guesses from training data** <-
- Empty — the model refuses to answer without context
- Identical to the RAG version


## Question 3: Token usage — short summary

### Summary Length: short

INFO 2026-07-03T01:37:11.926198Z 📊 Token Usage Summary:

Multilingual Agent:
- Input tokens: 282
- Output tokens: 77
- Total tokens: 359

English Brevity Agent:
- Input tokens: 92
- Output tokens: 34
- Total tokens: 126

💡 Tip: Monitor token usage to understand costs and optimize prompts!


### Question: What is the approximate output token count for multilingual_agent?

- 5-15 tokens
- -> **60-100 tokens** <-
- 200-400 tokens
- 500+ tokens

## Question 4: Token usage — long summary

### Summary Length: long

INFO 2026-07-03T01:41:04.982915Z 📊 Token Usage Summary:

Multilingual Agent:
- Input tokens: 282
- Output tokens: 187
- Total tokens: 469

English Brevity Agent:
- Input tokens: 202
- Output tokens: 44
- Total tokens: 246

💡 Tip: Monitor token usage to understand costs and optimize prompts!


In [2]:
short_summary_multilingual_agent_output_tokens = 77
long_summary_multilingual_agent_output_tokens = 187
long_summary_multilingual_agent_output_tokens / short_summary_multilingual_agent_output_tokens

2.4285714285714284

### Question: Compare the multilingual_agent output token count to your result from Question 3. Roughly how many times more output tokens does the long summary use?

- About the same (within 20%)
- -> **2-5x more** <-
- 10-20x more
- 50x more

## Question 5: Modifying a flow

### English Brevity Sentences: 3

INFO 2026-07-03T01:47:22.423149Z 📊 Token Usage Summary:

Multilingual Agent:
- Input tokens: 282
- Output tokens: 183
- Total tokens: 465

English Brevity Agent:
- Input tokens: 198
- Output tokens: 88
- Total tokens: 286

💡 Tip: Monitor token usage to understand costs and optimize prompts!


In [3]:
x1_sentence_english_brevity_agent_output_tokens = 44
x3_sentences_english_brevity_agent_output_tokens = 88
x3_sentences_english_brevity_agent_output_tokens / x1_sentence_english_brevity_agent_output_tokens

2.0

### Question: Compare the english_brevity output token count to the original 1-sentence version (also with summary_length = long). How do they compare?

- About the same (within 20%)
- -> **2-4x more** <-
- 5-10x more
- 10x+ more


## Question 6: Best Practices

### Question: Based on what you learned in this module, for production workflows requiring deterministic, repeatable results with strict compliance requirements (e.g., financial reporting, workflows in highly regulated industries), which approach is most appropriate?

- Always use AI agents for maximum flexibility and adaptation
- -> **Use traditional task-based workflows for predictability and auditability** <-
- Use only RAG without agents for better performance
- Use web search tools exclusively to ensure current data